# 📊 Market Insights — Agentic Workflow
**Stack:** Gemini Pro + yfinance + Langfuse  

An autonomous agent that executes a multi-step workflow without hand-holding:

```
Step 1 → Fetch real-time price data + momentum signals (yfinance)
Step 2 → Retrieve recent news headlines (yfinance news feed)
Step 3 → Lexicon-based sentiment analysis on headlines
Step 4 → LLM synthesis → structured market brief (Gemini Pro)
```

Every step is traced end-to-end in Langfuse with a `signal_alignment` eval score.

In [ ]:
!git clone https://github.com/Aurovindhya/ml-portfolio-suite.git
%cd ml-portfolio-suite
!pip install -q google-generativeai yfinance langfuse python-dotenv matplotlib

In [ ]:
import os
os.environ['GEMINI_API_KEY'] = 'your_gemini_api_key_here'
# os.environ['LANGFUSE_PUBLIC_KEY'] = 'pk-lf-...'
# os.environ['LANGFUSE_SECRET_KEY'] = 'sk-lf-...'

In [ ]:
# Step 1: Preview price data fetch (the agent's first tool call)
import sys; sys.path.insert(0, '.')
from modules.agentic.market_insights.model import _fetch_price_data
import json

ticker = 'AAPL'
price_data = _fetch_price_data(ticker)
print(f'Price data for {ticker}:')
print(json.dumps(price_data, indent=2))

In [ ]:
# Step 2: Preview sentiment analysis (the agent's third tool call)
from modules.agentic.market_insights.model import _analyze_sentiment

sample_headlines = [
    'Apple reports record quarterly revenue, beats analyst expectations',
    'iPhone sales surge in emerging markets',
    'Concerns over supply chain disruptions weigh on Apple outlook',
    'Apple stock rallies 3% after strong earnings report',
    'Analysts upgrade Apple to buy amid strong growth momentum',
]

sentiment = _analyze_sentiment(sample_headlines)
print('Sentiment analysis:')
print(json.dumps(sentiment, indent=2))

In [ ]:
# Full agentic run — all 4 steps, end-to-end
from modules.agentic.market_insights.model import run_market_agent

ticker = 'AAPL'   # try: TSLA, MSFT, NVDA, GOOGL
brief  = run_market_agent(ticker)

print('=' * 60)
print(f'  MARKET BRIEF: {brief["ticker"]} — {brief["company"]}')
print('=' * 60)
print(f'\nSummary: {brief["summary"]}')
print(f'\nSignals:')
print(f'  Price:     {brief["price_signal"]}')
print(f'  Sentiment: {brief["sentiment_signal"]}')
print(f'  Combined:  {brief["combined_signal"].upper()}')
print(f'\nKey catalysts:')
for cat in brief['key_catalysts']:
    print(f'  • {cat}')
print(f'\nRisks:')
for risk in brief['risks']:
    print(f'  ⚠ {risk}')
print(f'\nAnalyst note: {brief["analyst_note"]}')
print(f'\n[Trace: {brief["trace_id"]} | Total: {brief["total_ms"]:.0f}ms]')

In [ ]:
# Agent steps log — shows the agentic workflow execution trace
import pandas as pd

steps_df = pd.DataFrame(brief['steps_log'])
print('Agent execution log:')
print(steps_df.to_string(index=False))

In [ ]:
# Compare multiple tickers
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

tickers = ['AAPL', 'MSFT', 'TSLA']
results = {}

for t in tickers:
    try:
        results[t] = run_market_agent(t)
        print(f'{t}: {results[t]["combined_signal"].upper()} ({results[t]["total_ms"]:.0f}ms)')
    except Exception as e:
        print(f'{t}: Error — {e}')

# Signal summary chart
if results:
    signal_map = {'buy': 1, 'watch': 0.5, 'hold': 0, 'sell': -1}
    colors_map  = {'buy': 'green', 'watch': 'orange', 'hold': 'grey', 'sell': 'red'}
    labels = list(results.keys())
    signals = [results[t]['combined_signal'] for t in labels]
    values  = [signal_map.get(s, 0) for s in signals]
    colors  = [colors_map.get(s, 'grey') for s in signals]

    plt.figure(figsize=(8, 4))
    bars = plt.bar(labels, values, color=colors, edgecolor='black', linewidth=0.5)
    for bar, signal in zip(bars, signals):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 signal.upper(), ha='center', va='bottom', fontweight='bold')
    plt.axhline(0, color='black', linewidth=0.5)
    plt.title('Agent Signal Summary')
    plt.ylabel('Signal strength')
    plt.ylim(-1.3, 1.3)
    plt.tight_layout(); plt.show()